# Baseline EoH Run (HPC-Compatible)

This notebook runs original baseline EoH and supports school HPC LLM access via a local `/completions` bridge.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
EOH_SRC = ROOT / "eoh" / "src"
if EOH_SRC.exists() and str(EOH_SRC) not in sys.path:
    sys.path.insert(0, str(EOH_SRC))

print("ROOT:", ROOT)
print("EOH src in path:", str(EOH_SRC) in sys.path)


## Configuration

`LLM_MODE` options:
- `"hpc_bridge"`: uses your HPC OpenAI-compatible endpoint through a local bridge.
- `"direct_local"`: direct local `/completions` endpoint.
- `"direct_remote"`: direct remote endpoint through EoH API mode.

In [ ]:
LLM_MODE = "hpc_bridge"

# Shared run config
OUTPUT_PATH = "./"
POP_SIZE = 3
N_GENERATIONS = 2
N_PROC = 1
PROBLEM = "bp_online"  # "bp_online" or "tsp_construct"

# Mode: direct_local
LOCAL_URL = "http://127.0.0.1:11012/completions"

# Mode: direct_remote
REMOTE_API_ENDPOINT = "xxx"
REMOTE_API_KEY = "xxx"
REMOTE_MODEL = "gpt-3.5-turbo"

# Mode: hpc_bridge (reads env by default)
HPC_BASE = os.getenv("ENSIA_VLLM_BASE", "http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1")
HPC_API_KEY = os.getenv("ENSIA_VLLM_API_KEY", "")
HPC_MODEL = os.getenv("ENSIA_VLLM_MODEL", "auto")
BRIDGE_PORT = int(os.getenv("EOH_BRIDGE_PORT", "18000"))


In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
import requests

bridge_server = None
bridge_thread = None
EFFECTIVE_LOCAL_URL = LOCAL_URL
EFFECTIVE_MODEL = REMOTE_MODEL

if LLM_MODE == "hpc_bridge":
    if not HPC_API_KEY:
        raise RuntimeError("Missing ENSIA_VLLM_API_KEY (or set HPC_API_KEY in config cell).")

    auth_headers = {"Authorization": f"Bearer {HPC_API_KEY}"}
    model_id = HPC_MODEL
    if model_id == "auto":
        r = requests.get(f"{HPC_BASE}/models", headers=auth_headers, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f"/models failed: {r.status_code} {r.text[:300]}")
        payload = r.json()
        model_ids = [m.get("id") for m in payload.get("data", []) if m.get("id")]
        if not model_ids:
            raise RuntimeError("No model IDs returned by /models.")
        model_id = model_ids[0]
    EFFECTIVE_MODEL = model_id

    class BridgeHandler(BaseHTTPRequestHandler):
        def log_message(self, fmt, *args):
            return

        def _send(self, code, payload):
            body = json.dumps(payload).encode("utf-8")
            self.send_response(code)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_POST(self):
            if self.path != "/completions":
                self._send(404, {"error": "not found"})
                return
            try:
                length = int(self.headers.get("Content-Length", "0"))
                req = json.loads(self.rfile.read(length).decode("utf-8"))
                prompt = req.get("prompt", "")
                params = req.get("params", {}) or {}

                headers = {
                    "Authorization": f"Bearer {HPC_API_KEY}",
                    "Content-Type": "application/json",
                }

                chat_payload = {
                    "model": EFFECTIVE_MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": params.get("temperature", 0.2),
                }

                r = requests.post(f"{HPC_BASE}/chat/completions", headers=headers, json=chat_payload, timeout=600)
                if r.status_code == 200:
                    data = r.json()
                    text = data.get("choices", [{}])[0].get("message", {}).get("content")
                    if isinstance(text, str) and text:
                        self._send(200, {"content": [text]})
                        return

                comp_payload = {
                    "model": EFFECTIVE_MODEL,
                    "prompt": prompt,
                    "temperature": params.get("temperature", 0.2),
                    "max_tokens": params.get("max_new_tokens", 512),
                }
                r2 = requests.post(f"{HPC_BASE}/completions", headers=headers, json=comp_payload, timeout=600)
                if r2.status_code != 200:
                    self._send(502, {"error": "upstream", "chat_status": r.status_code, "comp_status": r2.status_code, "chat_text": r.text[:300], "comp_text": r2.text[:300]})
                    return
                data2 = r2.json()
                text2 = data2.get("choices", [{}])[0].get("text", "")
                self._send(200, {"content": [text2]})
            except Exception as e:
                self._send(500, {"error": str(e)})

    bridge_server = HTTPServer(("127.0.0.1", BRIDGE_PORT), BridgeHandler)
    bridge_thread = threading.Thread(target=bridge_server.serve_forever, daemon=True)
    bridge_thread.start()
    EFFECTIVE_LOCAL_URL = f"http://127.0.0.1:{BRIDGE_PORT}/completions"
    print("Bridge URL:", EFFECTIVE_LOCAL_URL)
    print("HPC base:", HPC_BASE)
    print("Model:", EFFECTIVE_MODEL)
elif LLM_MODE == "direct_local":
    EFFECTIVE_LOCAL_URL = LOCAL_URL
    EFFECTIVE_MODEL = REMOTE_MODEL
    print("Direct local URL:", EFFECTIVE_LOCAL_URL)
elif LLM_MODE == "direct_remote":
    EFFECTIVE_MODEL = REMOTE_MODEL
    print("Direct remote endpoint:", REMOTE_API_ENDPOINT)
else:
    raise ValueError("LLM_MODE must be one of: hpc_bridge, direct_local, direct_remote")


In [ ]:
if LLM_MODE in {"hpc_bridge", "direct_local"}:
    test = {
        "prompt": "Reply with exactly: OK",
        "repeat_prompt": 1,
        "params": {"do_sample": True}
    }
    r = requests.post(EFFECTIVE_LOCAL_URL, json=test, timeout=180)
    print(r.status_code)
    print(r.json())


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras

paras = Paras()
if LLM_MODE == "direct_remote":
    paras.set_paras(
        method="eoh",
        problem=PROBLEM,
        llm_use_local=False,
        llm_api_endpoint=REMOTE_API_ENDPOINT,
        llm_api_key=REMOTE_API_KEY,
        llm_model=EFFECTIVE_MODEL,
        ec_pop_size=POP_SIZE,
        ec_n_pop=N_GENERATIONS,
        exp_n_proc=N_PROC,
        exp_output_path=OUTPUT_PATH,
        exp_debug_mode=False,
    )
else:
    paras.set_paras(
        method="eoh",
        problem=PROBLEM,
        llm_use_local=True,
        llm_local_url=EFFECTIVE_LOCAL_URL,
        llm_model=EFFECTIVE_MODEL,
        ec_pop_size=POP_SIZE,
        ec_n_pop=N_GENERATIONS,
        exp_n_proc=N_PROC,
        exp_output_path=OUTPUT_PATH,
        exp_debug_mode=False,
    )

runner = eoh.EVOL(paras)
runner.run()


In [ ]:
import glob
import json
import os

pop_files = sorted(glob.glob("./results/pops/population_generation_*.json"))
best_files = sorted(glob.glob("./results/pops_best/population_generation_*.json"))

print("results exists:", os.path.isdir("./results"))
print("Population files:", pop_files)
print("Best files:", best_files)

if pop_files:
    latest = pop_files[-1]
    with open(latest, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Latest pop file:", latest)
    print("Individuals:", len(data))
    if len(data) > 0:
        print("Sample objective:", data[0].get("objective"))


In [ ]:
# Optional: stop bridge server when done.
if bridge_server is not None:
    bridge_server.shutdown()
    bridge_server.server_close()
    print("Bridge stopped.")
